In [1]:
for i in range(10,4,-1):
    print(i)

10
9
8
7
6
5


In [146]:
import pandas as pd
import numpy as np
import hnswlib
from time import time

In [147]:
df = pd.read_json("hf://datasets/mteb/index-NFCorpus-test-intfloat_e5-small-v2/embedding.jsonl", lines=True)

In [162]:
print(f"Count: {len(df)}")
print(df.keys())
print(f"Dim: {len(df.vector[0])}")
print(f"{df.vector[0][0]} - {type(df.vector[0][0])}")
# print(type(df.vector))
# print(type(df.vector[0]))
# print(type(np.asarray(df.vector)))
# dx = map(np.asarray, df.vector)
# print(type(dx))
# print(type(np.asarray(dx)))


# max([len(v) for v in df.vector])
# for v in df.vector:
#     print(len(v))

Count: 3633
Index(['id', 'vector', 'text'], dtype='str')
Dim: 384
-0.089692406356334 - <class 'float'>


In [178]:
class KNN:
    def __init__(self, data):
        self._data = data

    def euclidian_distance(x, y):
        sum = 0
        for i in range(0, min(len(x),len(y))):
            sum += (y[i] - x[i]) ** 2
        return np.sqrt(sum)

    def get_KNN(self, q, k, distance = euclidian_distance):
        dists = [distance(q, v) for v in self._data]
        knn_index = np.argsort(dists)[1:k+1]
        # return np.array([self._data[i] for i in knn_index])
        return np.array(knn_index)

In [ ]:
class HNSW_benchmark:
    def __init__(self, data, M = 16, ef_construction = 128):
        # alap adatok
        self._data = [np.array(v) for v in data]
        self._M = M
        self._ef_construction = ef_construction
        # KNN init
        self._knn = KNN(data)
        # HNSW init
        self._hnsw = hnswlib.Index(space        = 'l2',
                                   dim          = len(self._data[0]))
        self._hnsw.init_index(max_elements      = len(self._data),
                              M                 = self._M,
                              ef_construction   = self._ef_construction)
        self._hnsw.add_items(data               = self._data)

    def reconstruct_HNSW(self):
        self._hnsw.init_index(max_elements      = len(self._data),
                              M                 = self._M,
                              ef_construction   = self._ef_construction)
        self._hnsw.add_items(data               = self._data)

    def run(self, q, n_tries = 1000, k = 10, ef_search = 40):
        # Várt eredmény (recall = 1) kiszámítása
        R_E = self._knn.get_KNN(q, k)

        # HNSW keresési beállítások
        self._hnsw.set_ef(ef_search)

        # HNSW futtatása n_tries-szor, és recall számítása
        for i in range(0, n_tries):
            R_A = self._hnsw.knn_query([q], k, filter=())

    

In [183]:
# KNN & HNSW inicializálás
knn = KNN(df.vector)

hnsw = hnswlib.Index(space  = 'l2',
                     dim    = len(df.vector[0]))

hnsw.init_index(max_elements    = len(df.vector),
                M               = 16,
                ef_construction = 128)

# hnsw.add_items(data = [np.array(v) for v in df.vector],
#                ids  = df.id)
hnsw.add_items(data = [np.array(v) for v in df.vector])
# hnsw.add_items(data = [[1,2,3,4],[2,3,4,5],[3,4,5,6]])

In [186]:
q = df.vector[109]
k = 10

# KNN keresés
start = time()
res_knn = knn.get_KNN(q, k)
end = time()
# print(res_knn[0])
for i in range(0,k):
    # print(f"{i+1}: {KNN.euclidian_distance(q, df.vector[res_knn[i]])}") # ha id-ket kérünk
    # print(f"{i+1}: {df.id[res_knn[i]]}") # ha distance-eket kérünk
    print(f"{i+1}: {df.id[res_knn[i]]} at distance: {KNN.euclidian_distance(q, df.vector[res_knn[i]])}") # ha mindent kérünk
# print(list(res_knn))
print(f"{(end-start)*1000} ms")

#HNSW keresés
hnsw.set_ef(10)
start = time()
# res_hnsw = hnsw.knn_query([q], k+1)[0][0][1:k+1]
res_hnsw = hnsw.knn_query([q], k, filter=(lambda x: x != 109))[0][0]
end = time()
# print(res_hnsw[0])
# print(df.vector[res_hnsw[0]])
for i in range(0,k):
    # print(f"{i+1}: {KNN.euclidian_distance(q, df.vector[res_hnsw[i]])}") # ha id-ket kérünk
    # print(f"{i+1}: {df.id[res_hnsw[i]]}") # ha distance-eket kérünk
    print(f"{i+1}: {df.id[res_knn[i]]} at distance: {KNN.euclidian_distance(q, df.vector[res_hnsw[i]])}") # ha mindent kérünk
print(f"{(end-start)*1000} ms")

1: MED-4865 at distance: 0.4050179492495702
2: MED-4776 at distance: 0.4236859132497211
3: MED-2156 at distance: 0.4324077802443838
4: MED-1647 at distance: 0.44012787781145424
5: MED-869 at distance: 0.4417665856526358
6: MED-4777 at distance: 0.4442858366807119
7: MED-5231 at distance: 0.4456772726012128
8: MED-3942 at distance: 0.44695217536507387
9: MED-1570 at distance: 0.4479487434096029
10: MED-1864 at distance: 0.44797708975013467
543.0123805999756 ms
1: MED-4865 at distance: 0.4050179492495702
2: MED-4776 at distance: 0.4236859132497211
3: MED-2156 at distance: 0.4324077802443838
4: MED-1647 at distance: 0.44012787781145424
5: MED-869 at distance: 0.4417665856526358
6: MED-4777 at distance: 0.4442858366807119
7: MED-5231 at distance: 0.4456772726012128
8: MED-3942 at distance: 0.44695217536507387
9: MED-1570 at distance: 0.4479487434096029
10: MED-1864 at distance: 0.44797708975013467
0.43463706970214844 ms


In [ ]:
# print(np.argwhere(df.vector, q))
# print(df.vector.keys)

<bound method Series.keys of 0       [-0.089692406356334, 0.035149041563272004, 0.0...
1       [-0.08845881372690201, 0.040653113275766005, 0...
2       [-0.07727653533220201, 0.05864142253994901, 0....
3       [-0.10685417056083601, 0.023261113092303002, 0...
4       [-0.088368378579616, 0.094302326440811, 0.0228...
                              ...                        
3628    [-0.094853341579437, 0.013600675389170001, 0.0...
3629    [-0.07678522914648, 0.05750500410795201, 0.069...
3630    [-0.08102110028266901, 0.048747614026069, -0.0...
3631    [-0.043236788362264, 0.08697374165058101, 0.04...
3632    [-0.115256831049919, 0.04866611585021, 0.02988...
Name: vector, Length: 3633, dtype: object>
